# 🏋️ PulseFit · Notebook 02
## Bronze: bring the data in, exactly as it arrives

| 01 Set up | 02 Bronze | 03 Silver & Gold | 04 Answer the CEO | 05 Superpowers |
|:-:|:-:|:-:|:-:|:-:|
| ✅ done | 📍 **you are here** | ⬜ | ⬜ | ⬜ |

### Where we are
PulseFit's landing zone now holds the export files from all three systems. In this notebook you **load them into the
`BRONZE` layer exactly as they are**: no cleaning, no fixing.

This is the **E** and **L** of **ELT** (lecture: *"ETL → ELT"*). We **E**xtract and **L**oad first, then
**T**ransform later, inside the warehouse, in notebook 03.

**Why load "as-is"?** Bronze is our **replayable copy of the truth**. If we ever find a bug in our transformations,
we can rebuild everything from bronze without asking the source systems to send their data again.

## Step 0: Set the context
Notebooks don't remember where you were last time, so each one starts by choosing the role, warehouse,
database and schema.

> If you see an error such as *"does not exist or not authorized"*, run notebook **01** first.

In [ ]:
%%sql -r context
-- Every notebook starts by telling Snowflake WHO we are and WHERE we work
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE PULSEFIT_WH;
USE DATABASE PULSEFIT_DW;
USE SCHEMA BRONZE;

SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_DATABASE()  AS my_database,
       CURRENT_SCHEMA()    AS my_schema;

## Step 1: Look at the files
`LIST` shows every file in a stage, with its size.

In [ ]:
%%sql -r list_files
LIST @PULSEFIT_SOURCE.LANDING.LANDING_ZONE;

👀 **What to notice:** three folders, one per source system. Look closely at `door_scanners`: there is a file
called **`checkins_2025-09`**, although PulseFit's data starts on **1 October 2025**. Strange... keep that in mind,
it's a clue for notebook 03. 🕵️

## Step 2: Teach Snowflake how to read the files: file formats
A **file format** describes how to parse a file: CSV or JSON, is there a header row, how are values quoted, ...
We define it once and reuse it for every load.

In [ ]:
%%sql -r file_formats
CREATE OR REPLACE FILE FORMAT BRONZE.FF_CSV
    TYPE = CSV
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    TRIM_SPACE = FALSE
    EMPTY_FIELD_AS_NULL = TRUE
    COMMENT = 'CSV with a header row. Keep values exactly as sent (no trimming).';

CREATE OR REPLACE FILE FORMAT BRONZE.FF_JSON
    TYPE = JSON
    COMMENT = 'One JSON document per line';

SHOW FILE FORMATS IN SCHEMA BRONZE;

## Step 3: Peek inside the files before loading them
Snowflake can **query files directly in a stage** with SQL. `$1, $2, ...` are the columns of the file, and
`METADATA$FILENAME` tells you which file each row came from. Let's read Finance's price list.

In [ ]:
%%sql -r peek_price_list
SELECT $1                AS plan_code,
       $2                AS plan_name,
       $3                AS monthly_fee,
       $4                AS effective_from,
       METADATA$FILENAME AS file_name
FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/finance/ (FILE_FORMAT => 'BRONZE.FF_CSV');

👀 **What to notice**
* `MONTHLY_FEE` is **text with a comma** (`"8,000"`), not a number. That's what Excel exports.
* `EFFECTIVE_FROM` is `01/04/2026`. Is that 1 April or 4 January? (Finance uses day/month/year.)
* There are **two rows for STANDARD**: the old price and the new one. That's the price increase from the story.

We will **not** fix any of this now. Bronze keeps the data exactly as sent; fixing is for silver.

Now one of the door-scanner files. Each line is one JSON event.

In [ ]:
%%sql -r peek_checkins
SELECT $1                AS raw_event,
       METADATA$FILENAME AS file_name
FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/door_scanners/ (FILE_FORMAT => 'BRONZE.FF_JSON')
LIMIT 5;

👀 **What to notice:** each event is nested. It has a `device` object (which club and door) and a `session`
object containing an **array** of `activities`. The scanner knows a **`card_id`**, not a member id, and the time
`event_ts` ends in **`Z`**, meaning UTC (not Sri Lankan time).

## Step 4: Create the bronze tables
Three design choices, all very common in real projects:

1. **CSV columns are all `STRING`.** A load should never fail because one value looks odd (like `"8,000"`).
   We convert to proper types in silver.
2. **JSON goes into a single `VARIANT` column.** `VARIANT` is Snowflake's type for semi-structured data:
   store the whole document now, decide which fields matter later (*schema-on-read*).
3. **Every table gets two audit columns:** `_SOURCE_FILE` (which file did this row come from?) and
   `_LOADED_AT` (when?). This is called **lineage**, and it is gold when you need to debug.

In [ ]:
%%sql -r create_bronze_tables
CREATE OR REPLACE TABLE BRONZE.CLUBS (
    club_id STRING, club_code STRING, club_name STRING, city STRING, province STRING, opened_on STRING,
    _source_file STRING, _loaded_at TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.PRICE_LIST (
    plan_code STRING, plan_name STRING, monthly_fee STRING, effective_from STRING,
    _source_file STRING, _loaded_at TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.MEMBER_SNAPSHOTS (
    snapshot_date STRING, member_id STRING, card_id STRING, first_name STRING, last_name STRING,
    email STRING, birth_year STRING, home_club_id STRING, plan_code STRING, status STRING, joined_on STRING,
    _source_file STRING, _loaded_at TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.CHECKIN_EVENTS (
    payload VARIANT,
    _source_file STRING, _loaded_at TIMESTAMP_LTZ);

SHOW TABLES IN SCHEMA BRONZE;

## Step 5: Load the files with `COPY INTO`
`COPY INTO` is Snowflake's bulk loader. It reads many files **in parallel** and can do small transformations on
the way in. Here we add the file name and load time to every row. Let's start with the two small reference files.

In [ ]:
%%sql -r copy_reference_files
COPY INTO BRONZE.CLUBS
FROM (SELECT $1, $2, $3, $4, $5, $6, METADATA$FILENAME, CURRENT_TIMESTAMP()
      FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/member_app/clubs/)
FILE_FORMAT = (FORMAT_NAME = 'BRONZE.FF_CSV');

COPY INTO BRONZE.PRICE_LIST
FROM (SELECT $1, $2, $3, $4, METADATA$FILENAME, CURRENT_TIMESTAMP()
      FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/finance/)
FILE_FORMAT = (FORMAT_NAME = 'BRONZE.FF_CSV');

👀 `COPY INTO` returns one row **per file**: its status, how many rows it parsed and loaded, and any errors.

Next, the ten monthly member exports, in one command:

In [ ]:
%%sql -r copy_member_exports
COPY INTO BRONZE.MEMBER_SNAPSHOTS
FROM (SELECT $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, METADATA$FILENAME, CURRENT_TIMESTAMP()
      FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/member_app/members/)
FILE_FORMAT = (FORMAT_NAME = 'BRONZE.FF_CSV');

And the door-scanner events. `$1` is the whole JSON document, which lands in the `VARIANT` column:

In [ ]:
%%sql -r copy_checkin_events
COPY INTO BRONZE.CHECKIN_EVENTS
FROM (SELECT $1, METADATA$FILENAME, CURRENT_TIMESTAMP()
      FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/door_scanners/)
FILE_FORMAT = (FORMAT_NAME = 'BRONZE.FF_JSON');

## Step 6: Run it again! (rerunnable loads)
Pipelines fail and get re-run all the time. A **rerunnable** (idempotent) load gives the same result whether you
run it once or five times (lecture: *"Rerunnable (idempotent) pipelines"*).

Snowflake remembers **which files it has already loaded into each table** (for 64 days). Let's run exactly the
same `COPY INTO` again:

In [ ]:
%%sql -r copy_checkin_events_again
COPY INTO BRONZE.CHECKIN_EVENTS
FROM (SELECT $1, METADATA$FILENAME, CURRENT_TIMESTAMP()
      FROM @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/door_scanners/)
FILE_FORMAT = (FORMAT_NAME = 'BRONZE.FF_JSON');

👀 **What to notice:** *"Copy executed with 0 files processed"*. No duplicates. When the scanners drop a
new file tomorrow, this same command will load **only the new file**.

## Step 7: What's in bronze now?

In [ ]:
%%sql -r bronze_row_counts
SELECT 'CLUBS'            AS table_name, COUNT(*) AS row_count FROM BRONZE.CLUBS
UNION ALL
SELECT 'PRICE_LIST',                     COUNT(*)              FROM BRONZE.PRICE_LIST
UNION ALL
SELECT 'MEMBER_SNAPSHOTS',               COUNT(*)              FROM BRONZE.MEMBER_SNAPSHOTS
UNION ALL
SELECT 'CHECKIN_EVENTS',                 COUNT(*)              FROM BRONZE.CHECKIN_EVENTS;

👀 About **210,000 check-in events** and **21,800 member rows** (2,400 members × up to 10 monthly exports).
Small for Snowflake, but already far too much for a spreadsheet.

## Step 8: Query JSON with plain SQL
You don't need to convert JSON into tables before you can query it:
* `payload:event_id` reads a field, and `payload:device.club_code` reads a nested field
* `::STRING`, `::INT`, `::TIMESTAMP_NTZ` convert (*cast*) the value to a proper type

In [ ]:
%%sql -r query_json
SELECT payload:event_id::STRING             AS event_id,
       payload:event_ts::TIMESTAMP_NTZ      AS event_time_utc,
       payload:card_id::STRING              AS card_id,
       payload:device.club_code::STRING     AS club_code,
       payload:session.duration_min::INT    AS duration_min,
       payload:session.activities           AS activities
FROM BRONZE.CHECKIN_EVENTS
LIMIT 10;

`activities` is an **array**: one visit can include cardio *and* weights. `FLATTEN` turns an array into one row
per element, so we can count which activities are most popular:

In [ ]:
%%sql -r popular_activities
SELECT a.value::STRING AS activity,
       COUNT(*)        AS sessions
FROM BRONZE.CHECKIN_EVENTS e,
     LATERAL FLATTEN(input => e.payload:session.activities) a
GROUP BY activity
ORDER BY sessions DESC;

💡 **Try it:** the result grid has a built-in **chart** option. Switch it on to turn this result into a bar chart
without writing any code.

## Step 9: The audit trail
Every load is recorded. `COPY_HISTORY` shows which file was loaded, when, and how many rows it had, which is
exactly what you need when someone asks *"did yesterday's file arrive?"*

In [ ]:
%%sql -r load_history
SELECT table_name, file_name, status, row_count, last_load_time
FROM TABLE(PULSEFIT_DW.INFORMATION_SCHEMA.COPY_HISTORY(
         TABLE_NAME => 'PULSEFIT_DW.BRONZE.CHECKIN_EVENTS',
         START_TIME => DATEADD(hour, -24, CURRENT_TIMESTAMP())))
ORDER BY file_name;

## ✅ What you did
* Defined **file formats** and **queried files directly** in a stage before loading.
* Created **bronze tables**: strings for CSV, `VARIANT` for JSON, plus lineage columns.
* Bulk-loaded every file with **`COPY INTO`**, and saw that re-running it loads **nothing twice**.
* Queried **semi-structured JSON** with plain SQL and `FLATTEN`.

## 🤔 Think about it
1. Bronze still contains prices like `"8,000"` and duplicate events. Why is it a *good* idea **not** to fix them
   while loading?
2. What would go wrong if `COPY INTO` did **not** remember which files it had loaded, and the nightly job ran twice?

## ➡️ Next
Open **`03_Silver_Gold_Transform_and_Model`**. You'll clean the data and build the star schema.